In [1]:
#!unzip data.zip


# 🛣️ نظام كشف الحفر - Pothole Detection System
**شغّلي كل cell بالترتيب من فوق لتحت** ✅

---
## الخطوات:
1. تثبيت المكتبات
2.  رفع الداتا
3. تجهيز الداتا
4. تدريب الموديل
5. كشف الحفر
6. حفظ النتائج في Excel
7. نظام التنبيهات
8. تنزيل النتائج

## 📦 الخطوة 1 - تثبيت المكتبات

In [2]:
!pip install ultralytics opencv-python pandas openpyxl -q
print('✅ المكتبات اتثبتت')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 27.9 MB/s eta 0:00:00
✅ المكتبات اتثبتت


## ☁️ الخطوة 2 - وصّلي Google Drive

> ⚠️ **مهم:** صورك وداتاك لازم تكون على Google Drive
> لو مش عندك Drive هترفعي الملفات يدوي في الخطوة الجاية

## 📁 الخطوة 3 - تجهيز الداتا

> ✏️ **غيّري المسارين دول** حسب مكان ملفاتك على Drive

In [3]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive mounted successfully!')

Mounted at /content/drive
✅ Google Drive mounted successfully!


After running the above cell and authorizing Google Drive access, your Drive contents will be available under `/content/drive/MyDrive/`. You can then navigate through this directory to find your `images` and `annotations_txt` folders.

For example, if your `data.zip` file is in a folder called `pothole_dataset` directly in your Google Drive, the paths might look like this:

`images_path = '/content/drive/MyDrive/pothole_dataset/images'`
`labels_path = '/content/drive/MyDrive/pothole_dataset/annotations_txt'`

Please update the `images_path` and `labels_path` variables in the cell `gmAcvKRJmNz3` with the correct paths after you've mounted your Drive. After updating, re-run cell `gmAcvKRJmNz3`.

In [4]:
import os

# List the contents of your Google Drive's MyDrive folder
my_drive_path = '/content/drive/MyDrive/graduation digi/test'

if os.path.exists(my_drive_path):
    print(f"Contents of {my_drive_path}:")
    for item in os.listdir(my_drive_path):
        print(f"- {item}")
else:
    print(f"❌ {my_drive_path} does not exist. Make sure Google Drive is mounted correctly and the path is correct.")


❌ /content/drive/MyDrive/graduation digi/test does not exist. Make sure Google Drive is mounted correctly and the path is correct.


In [5]:
import os, shutil, random

# ✏️ غيّري المسارين دول
images_path = '/content/drive/MyDrive/graduation digi/test/images'        # فولدر الصور
labels_path = '/content/drive/MyDrive/graduation digi/test/annotations_txt'  # فولدر الـ labels

# ── التحقق من وجود الملفات ───────────────────────────
if not os.path.exists(images_path):
    print(f'❌ مش لاقي الصور في: {images_path}')
    print('غيّري المسار في السطر اللي فوق')
elif not os.path.exists(labels_path):
    print(f'❌ مش لاقي الـ labels في: {labels_path}')
    print('غيّري المسار في السطر اللي فوق')
else:
    # Clear existing dataset directory if it exists to ensure a clean run
    if os.path.exists('dataset'):
        shutil.rmtree('dataset')
        print("🗑️ Removed existing 'dataset' directory for a clean setup.")

    # ── إنشاء الفولدرات ───────────────────────────────
    for folder in ['dataset/train/images', 'dataset/train/labels',
                   'dataset/val/images',   'dataset/val/labels',
                   'outputs']:
        os.makedirs(folder, exist_ok=True)

    # ── نقل الصور ─────────────────────────────────────
    images = [f for f in os.listdir(images_path) if f.endswith(('.jpg', '.png'))]
    print(f'عدد الصور: {len(images)}')

    copied, skipped = 0, 0
    for img in images:
        txt      = img.replace('.jpg', '.txt').replace('.png', '.txt')
        img_src  = os.path.join(images_path, img)
        txt_src  = os.path.join(labels_path, txt)

        if not os.path.exists(txt_src):
            skipped += 1
            continue

        split = 'train' if random.random() < 0.8 else 'val'
        shutil.copy(img_src, f'dataset/{split}/images/{img}')
        shutil.copy(txt_src, f'dataset/{split}/labels/{txt}')
        copied += 1

    train_n = len(os.listdir('dataset/train/images'))
    val_n   = len(os.listdir('dataset/val/images'))
    print(f'Train: {train_n} صورة  |  Val: {val_n} صورة')
    print(f'تم نقل: {copied}  |  مش ليها label: {skipped}')

    # --- Add explicit verification step ---
    print("\n--- Verifying Dataset Setup ---")
    base_dataset_path = 'dataset'
    if os.path.exists(base_dataset_path):
        print(f"✅ Base dataset directory '{base_dataset_path}' exists.")
        all_checks_passed = True

        for sub_path in ['train/images', 'train/labels', 'val/images', 'val/labels']:
            full_path = os.path.join(base_dataset_path, sub_path)
            if os.path.exists(full_path) and len(os.listdir(full_path)) > 0:
                print(f"  ✅ '{full_path}' exists and contains {len(os.listdir(full_path))} files.")
            else:
                print(f"  ❌ Error: '{full_path}' is missing or empty.")
                all_checks_passed = False

        if all_checks_passed:
            print('✅ Dataset جاهز')
        else:
            print('❌ Dataset setup incomplete. Please review previous steps.')

    else:
        print(f"❌ Error: The base dataset directory '{base_dataset_path}' does not exist after creation attempts.")
        print('❌ Dataset setup failed.')


❌ مش لاقي الصور في: /content/drive/MyDrive/graduation digi/test/images
غيّري المسار في السطر اللي فوق


### Re-verifying Google Drive Folder Contents

Let's re-check the contents of your `graduation digi/test` folder to ensure the `images` and `annotations_txt` directories are still present and accessible.

In [6]:
import os

# List the contents of your Google Drive's specific folder
check_path = '/content/drive/MyDrive/graduation digi/test'

if os.path.exists(check_path):
    print(f"Contents of {check_path}:")
    contents = os.listdir(check_path)
    for item in contents:
        print(f"- {item}")

    if 'images' in contents and 'annotations_txt' in contents:
        print("✅ Both 'images' and 'annotations_txt' folders are found in the specified path.")
    else:
        print("❌ One or both of 'images' or 'annotations_txt' folders are missing from the specified path.")
        print("Please ensure these folders exist directly within '/content/drive/MyDrive/graduation digi/test'.")
else:
    print(f"❌ {check_path} does not exist. Please double-check the path and ensure Google Drive is mounted correctly.")


❌ /content/drive/MyDrive/graduation digi/test does not exist. Please double-check the path and ensure Google Drive is mounted correctly.


In [7]:
import os

my_drive_root = '/content/drive/MyDrive'

if os.path.exists(my_drive_root):
    print(f"Contents of {my_drive_root}:")
    for item in os.listdir(my_drive_root):
        print(f"- {item}")
else:
    print(f"❌ {my_drive_root} does not exist. Please ensure Google Drive is mounted correctly.")


Contents of /content/drive/MyDrive:
- Colab Notebooks
- Saved from Chrome
- portfolio
- graduation digi
- wesam


## ⚙️ الخطوة 4 - إنشاء data.yaml

In [8]:
import yaml, os

data = {
    'path': os.path.abspath('dataset'),  # المسار الكامل عشان Colab
    'train': 'train/images',
    'val':   'val/images',
    'nc':    1,
    'names': ['pothole']
}

with open('data.yaml', 'w') as f:
    yaml.dump(data, f, default_flow_style=False)

print('✅ data.yaml اتعمل')
print(f'   path: {data["path"]}')

✅ data.yaml اتعمل
   path: /content/dataset


## 🧠 الخطوة 5 - تدريب الموديل

> ⏳ هياخد وقت - ممكن 10-30 دقيقة حسب حجم الداتا
> شغّلي GPU من **Runtime → Change runtime type → T4 GPU** عشان يبقى أسرع

In [9]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')  # ابدأ من الموديل الأساسي

model.train(
    data='data.yaml',
    epochs=100,
    imgsz=640,
    batch=16,      # 16 على GPU - غيّريها لـ 8 لو على CPU
    patience=10,   # يوقف تلقائياً لو مفيش تحسن
    save=True,
    plots=True,
)

print('✅ التدريب خلص')

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.68 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=Fals

RuntimeError: Dataset 'data.yaml' error ❌ Dataset 'data.yaml' images not found, missing path '/content/dataset/val/images'
Note dataset download directory is '/content/datasets'. You can update this in '/root/.config/Ultralytics/settings.json'

### Verify Dataset Directory Contents

Before re-attempting training, let's confirm that the `dataset` directory was correctly populated with images and labels.

In [ ]:
import os

# Check if the base dataset directory exists
base_dataset_path = 'dataset'
if not os.path.exists(base_dataset_path):
    print(f"❌ Error: The base dataset directory '{base_dataset_path}' does not exist.")
else:
    print(f"✅ Base dataset directory '{base_dataset_path}' exists.")

# Check the contents of the 'train' image and label directories
train_images_path = os.path.join(base_dataset_path, 'train', 'images')
train_labels_path = os.path.join(base_dataset_path, 'train', 'labels')

if os.path.exists(train_images_path):
    num_train_images = len(os.listdir(train_images_path))
    print(f"  Train images found at '{train_images_path}': {num_train_images} files.")
else:
    print(f"❌ Train images directory '{train_images_path}' does not exist.")

if os.path.exists(train_labels_path):
    num_train_labels = len(os.listdir(train_labels_path))
    print(f"  Train labels found at '{train_labels_path}': {num_train_labels} files.")
else:
    print(f"❌ Train labels directory '{train_labels_path}' does not exist.")

# Check the contents of the 'val' image and label directories
val_images_path = os.path.join(base_dataset_path, 'val', 'images')
val_labels_path = os.path.join(base_dataset_path, 'val', 'labels')

if os.path.exists(val_images_path):
    num_val_images = len(os.listdir(val_images_path))
    print(f"  Validation images found at '{val_images_path}': {num_val_images} files.")
else:
    print(f"❌ Validation images directory '{val_images_path}' does not exist.")

if os.path.exists(val_labels_path):
    num_val_labels = len(os.listdir(val_labels_path))
    print(f"  Validation labels found at '{val_labels_path}': {num_val_labels} files.")
else:
    print(f"❌ Validation labels directory '{val_labels_path}' does not exist.")


## ✅ الخطوة 6 - تأكد إن الموديل اتحفظ

In [ ]:
import os

model_path = 'runs/detect/train-2/weights/best.pt'

if os.path.exists(model_path):
    size = os.path.getsize(model_path) / 1024 / 1024
    print(f'✅ الموديل موجود: {model_path}')
    print(f'   الحجم: {size:.1f} MB')
else:
    print('❌ الموديل مش موجود - شوفي error في التدريب')

## 🔍 الخطوة 7 - كشف الحفر في صورة جديدة

In [ ]:
# ارفعي صورة الحفرة
from google.colab import files
print('اختاري صورة الحفرة...')
uploaded = files.upload()
image_name = list(uploaded.keys())[0]
print(f'✅ الصورة: {image_name}')

In [ ]:
from ultralytics import YOLO
import cv2
from IPython.display import display, Image as IPImage

# ── تحميل الموديل المدرّب ──────────────────────────────
model = YOLO('runs/detect/train-2/weights/best.pt')

# ── كشف الحفر ─────────────────────────────────────────
results = model.predict(
    source=image_name,
    conf=0.3,
    verbose=False
)

result = results[0]

# ── عرض النتائج ───────────────────────────────────────
print(f'عدد الحفر المكتشفة: {len(result.boxes)}')
print('─' * 40)

detections = []
for i, box in enumerate(result.boxes):
    conf       = float(box.conf[0])
    x1,y1,x2,y2 = box.xyxy[0].tolist()
    area       = (x2 - x1) * (y2 - y1)

    # تحديد الخطورة
    if area > 15000:
        severity = 'high'
        emoji    = '🔴'
    elif area > 5000:
        severity = 'medium'
        emoji    = '🟠'
    else:
        severity = 'low'
        emoji    = '🟢'

    detections.append({
        'confidence': round(conf, 4),
        'area':       round(area, 2),
        'severity':   severity
    })

    print(f'  حفرة {i+1}: ثقة={conf:.2f} | مساحة={area:.0f}px | {emoji} {severity}')

# ── حفظ وعرض الصورة بالـ boxes ───────────────────────
annotated = result.plot()
cv2.imwrite('outputs/result.jpg', annotated)
print('─' * 40)
print('✅ الصورة اتحفظت في outputs/result.jpg')
display(IPImage('outputs/result.jpg'))

## 📊 الخطوة 8 - حفظ النتائج في Excel

In [ ]:
import pandas as pd
from datetime import datetime
from pathlib import Path

excel_path = 'outputs/violations.xlsx'
now        = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

# ── تجهيز الصفوف الجديدة ──────────────────────────────
new_rows = [{
    'time':       now,
    'type':       'pothole',
    'confidence': d['confidence'],
    'area':       d['area'],
    'severity':   d['severity']
} for d in detections]

new_df = pd.DataFrame(new_rows)

# ── Append (مش overwrite) ─────────────────────────────
if Path(excel_path).exists():
    old_df = pd.read_excel(excel_path)
    final  = pd.concat([old_df, new_df], ignore_index=True)
else:
    final  = new_df

final.to_excel(excel_path, index=False)

print(f'✅ النتائج اتحفظت في {excel_path}')
print(f'   إجمالي السجلات: {len(final)}')
print()
print(final.tail(10).to_string(index=False))

## ⚠️ الخطوة 9 - نظام التنبيهات

In [ ]:
import pandas as pd
from datetime import datetime
from pathlib import Path
import os

excel_path = 'outputs/violations.xlsx'

# Ensure the 'outputs' directory exists
Path('outputs').mkdir(parents=True, exist_ok=True)

# Check if 'detections' variable is defined
if 'detections' not in locals() and 'detections' not in globals():
    print("❌ Error: The 'detections' variable is not defined.")
    print("   Please ensure you have run the detection cell (Step 7: `qob-Y3C9mNz7`) first to generate detections for the current image.")
    # Set default values to prevent further errors and allow the cell to complete
    total_current_image = 0
    high_count_current_image = 0
    med_count_current_image = 0
    low_count_current_image = 0
    alert_message_current_image = 'لا توجد حفر مكتشفة لأنه لم يتم تشغيل الكشف.'
else:
    # These counts are for the *current image's* detections (from the 'detections' variable)
    # 'detections' variable is populated in cell qob-Y3C9mNz7
    total_current_image      = len(detections)
    high_count_current_image = sum(1 for d in detections if d['severity'] == 'high')
    med_count_current_image  = sum(1 for d in detections if d['severity'] == 'medium')
    low_count_current_image  = sum(1 for d in detections if d['severity'] == 'low')

    # Collect alert messages for the *current image's* summary
    alerts_current_image = []
    if high_count_current_image > 0:
        alerts_current_image.append(f'CRITICAL: {high_count_current_image} حفرة خطيرة - يحتاج تدخل فوري!')
    if total_current_image > 3:
        alerts_current_image.append(f'WARNING: {total_current_image} حفرة في الصورة - الطريق يحتاج صيانة')
    if high_count_current_image == 0 and total_current_image <= 3:
        alerts_current_image.append(f'INFO: الطريق في حالة مقبولة')
    alert_message_current_image = '; '.join(alerts_current_image)


# --- Print summary to console ---
print('\n── نتيجة الفحص ─────────────────────────')
print(f'  إجمالي الحفر    : {total_current_image}')
print(f'  🔴 خطيرة        : {high_count_current_image}')
print(f'  🟠 متوسطة       : {med_count_current_image}')
print(f'  🟢 بسيطة        : {low_count_current_image}')
print('── التنبيهات ────────────────────────────')

if high_count_current_image > 0:
    print(f'🚨 CRITICAL: {high_count_current_image} حفرة خطيرة - يحتاج تدخل فوري!')
if total_current_image > 3:
    print(f'⚠️  WARNING: {total_current_image} حفرة في الصورة - الطريق يحتاج صيانة')
if high_count_current_image == 0 and total_current_image <= 3:
    print(f'✅ INFO: الطريق في حالة مقبولة')

print('─────────────────────────────────────────')

# --- Prepare and write summary DataFrame to Excel ---

# Create a single row DataFrame for the current image's summary
current_image_summary_row = pd.DataFrame([{
    'Timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'Total Potholes (Image)': total_current_image,
    'High Severity (Image)': high_count_current_image,
    'Medium Severity (Image)': med_count_current_image,
    'Low Severity (Image)': low_count_current_image,
    'Image Status Message': alert_message_current_image
}])

# Try to read existing summary sheet, or initialize an empty DataFrame
existing_summary_df = pd.DataFrame()
if Path(excel_path).exists():
    try:
        existing_summary_df = pd.read_excel(excel_path, sheet_name='Summary')
    except ValueError:
        pass # No 'Summary' sheet yet, continue with empty df

# Concatenate the new summary row with existing summaries
updated_summary_df = pd.concat([existing_summary_df, current_image_summary_row], ignore_index=True)

# Read the latest individual detections data from the Excel file (saved in previous cell NxPLjNpnmNz7)
df_latest_detections = pd.DataFrame()
if Path(excel_path).exists():
    try:
        df_latest_detections = pd.read_excel(excel_path, sheet_name='Detections')
    except ValueError:
        pass # No 'Detections' sheet yet, start with empty df

print(f'\n📊 إجمالي السجلات الفردية في Excel: {len(df_latest_detections)}')

# Write to Excel, updating both 'Detections' and 'Summary' sheets
# Use 'openpyxl' engine to handle multiple sheets.
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    # Write the full individual detections to the 'Detections' sheet
    df_latest_detections.to_excel(writer, sheet_name='Detections', index=False)
    # Write the updated summary data to the 'Summary' sheet
    updated_summary_df.to_excel(writer, sheet_name='Summary', index=False)

print(f'✅ تم تحديث {excel_path} بورقتي \'Detections\' و \'Summary\'.')
print(f'   آخر إضافة لـ \'Summary\':')
print(current_image_summary_row.to_string(index=False))

In [ ]:
import pandas as pd

excel_path = 'outputs/violations.xlsx'

print(f"\nContent of the 'Summary' sheet in {excel_path}:\n")
try:
    summary_df = pd.read_excel(excel_path, sheet_name='Summary')
    display(summary_df)
except ValueError:
    print("The 'Summary' sheet does not exist in the Excel file.")
except FileNotFoundError:
    print(f"The file {excel_path} was not found.")


## 💾 الخطوة 10 - تنزيل النتائج

In [ ]:
from google.colab import files

print('جاري التنزيل...')
files.download('outputs/result.jpg')       # الصورة بالـ boxes
files.download('outputs/violations.xlsx')  # ملف Excel
print('✅ تم التنزيل')

In [ ]:
import json
from datetime import datetime

# تحويل النتائج لـ JSON جاهز لـ n8n
output_json = []

for i, d in enumerate(detections):

    # تحديد الخطورة بالعربي
    if d['severity'] == 'high':
        details = 'حفرة خطيرة - تحتاج تدخل فوري'
    elif d['severity'] == 'medium':
        details = 'حفرة متوسطة - تحتاج متابعة'
    else:
        details = 'حفرة بسيطة - تحتاج مراقبة'

    violation = {
        "type":       "Violation",
        "location":   "Main St",        # ✏️ غيّريها لو عندك GPS
        "details":    details,
        "severity":   d['severity'],
        "confidence": d['confidence'],
        "area":       d['area'],
        "timestamp":  datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    }
    output_json.append(violation)

# حفظ الـ JSON
with open('outputs/violations.json', 'w', encoding='utf-8') as f:
    json.dump(output_json, f, ensure_ascii=False, indent=2)

# عرض النتيجة
print('✅ JSON جاهز لـ n8n:')
print(json.dumps(output_json, ensure_ascii=False, indent=2))

In [ ]:
import requests
import json

# ✏️ حطي هنا الـ webhook URL بتاعك من n8n (تأكدي إنه URL عام مش localhost)
WEBHOOK_URL = "http://localhost:5678/workflow/GWLeq9oIQzixycOx?projectId=GS6n8tubmhYytXIN"

# بعت الـ JSON لـ n8n
for violation in output_json:
    response = requests.post(
        WEBHOOK_URL,
        json=violation,
        headers={"Content-Type": "application/json"}
    )
    print(f"✅ اتبعت: {response.status_code} | {violation['severity']}")

### اختياري: اختبار الـ Webhook URL

تقدري تستخدمي الـ function دي عشان تتأكدي إن الـ `WEBHOOK_URL` اللي حاطاه صح و n8n شغال وبيستقبل الـ requests.

In [ ]:
import requests

def test_webhook_connection(url, payload=None):
    """Tests a webhook connection by sending a POST request."""
    print(f"Testing connection to: {url}")
    try:
        # Send a dummy POST request to test the connection
        # Using a simple dictionary as a payload
        if payload is None:
            payload = {"test": "hello from Colab"}

        response = requests.post(url, json=payload, headers={"Content-Type": "application/json"}, timeout=5)

        print(f"Response Status Code: {response.status_code}")
        if response.ok:
            print("✅ Webhook connection successful!")
        else:
            print(f"⚠️ Webhook returned an error: {response.text}")

    except requests.exceptions.ConnectionError as e:
        print(f"❌ Connection Error: The server at {url} refused the connection.")
        print("   Make sure n8n is running and the URL is publicly accessible, not localhost.")
    except requests.exceptions.Timeout:
        print(f"❌ Timeout Error: The request to {url} timed out.")
        print("   The server might be slow or unreachable.")
    except requests.exceptions.RequestException as e:
        print(f"❌ An unexpected error occurred: {e}")

# ✏️ استبدلي WEBHOOK_URL هنا بالرابط الصحيح بتاعك عشان تجربي
# test_webhook_connection(WEBHOOK_URL)
# For example, if you have a correct public URL, you could use:
# test_webhook_connection("https://your-n8n-instance.com/webhook/your-workflow-id")

print("Run the function above with your actual webhook URL to test the connection.")